# Backtesting

## Package: backtesting
- https://kernc.github.io/backtesting.py/doc/backtesting/#gsc.tab=0


In [ ]:
%%capture

!pip install backtesting

In [ ]:
from backtesting import Backtest, Strategy

### Example 1: Crossover of SMA10 and SMA20 for GOOG

In [ ]:
from backtesting.test import SMA
from backtesting.lib import crossover

class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)

    def next(self):
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            self.sell()

In [ ]:
from backtesting.test import GOOG

bt = Backtest(GOOG, SmaCross, commission = .002, exclusive_orders = True)
stats = bt.run()
bt.plot()

In [ ]:
stats

### Example 2

In [ ]:
import ffn

tsmc = ffn.get("2330.tw:Open, 2330.tw:High, 2330.tw:Low, 2330.tw:Close", start = "2020-01-01")
tsmc.columns = [x[6].upper() + x[7:] for x in tsmc.columns]
tsmc

In [ ]:
class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 5)
        self.ma2 = self.I(SMA, price, 10)

    def next(self):
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            self.sell()

In [ ]:
bt = Backtest(tsmc, SmaCross, commission = .002, exclusive_orders = True)
stats = bt.run()
bt.plot()

### Example 3: Multiple Time Frames
- https://kernc.github.io/backtesting.py/doc/examples/Multiple%20Time%20Frames.html

### Example 4: Parameter Heatmap

In [ ]:
class Sma4Cross(Strategy):

    n1 = 50
    n2 = 100
    n_enter = 20
    n_exit = 10
    
    def init(self):
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
        self.sma_enter = self.I(SMA, self.data.Close, self.n_enter)
        self.sma_exit = self.I(SMA, self.data.Close, self.n_exit)
        
    def next(self):
        
        if not self.position:
            
            # On upwards trend, if price closes above
            # "entry" MA, go long
            
            # Here, even though the operands are arrays, this
            # works by implicitly comparing the two last values
            if self.sma1 > self.sma2:
                if crossover(self.data.Close, self.sma_enter):
                    self.buy()
                    
            # On downwards trend, if price closes below
            # "entry" MA, go short
            
            else:
                if crossover(self.sma_enter, self.data.Close):
                    self.sell()
        
        # But if we already hold a position and the price
        # closes back below (above) "exit" MA, close the position
        
        else:
            if (self.position.is_long and crossover(self.sma_exit, self.data.Close)
               or self.position.is_short and crossover(self.data.Close, self.sma_exit)):
                self.position.close()

In [ ]:
backtest = Backtest(GOOG, Sma4Cross, commission = .002)

stats, heatmap = backtest.optimize(n1 = range(10, 110, 10),
                   n2 = range(20, 210, 20),
                   n_enter = range(15, 35, 5),
                   n_exit = range(10, 25, 5),
                   constraint = lambda p: p.n_exit < p.n_enter < p.n1 < p.n2,
                   maximize = "Equity Final [$]",
                   max_tries = 200,
                   return_heatmap = True)

In [ ]:
hm = heatmap.groupby(["n1", "n2"]).mean().unstack()
hm

In [ ]:
%matplotlib inline

import seaborn as sns

sns.heatmap(hm[::-1], cmap = "viridis")

In [ ]:
from backtesting.lib import plot_heatmaps

plot_heatmaps(heatmap, agg = "mean")

### Example 5: Model-Based Optimization

In [ ]:
%%capture

! pip install scikit-optimize  # This is a run-time dependency

In [ ]:
%%time

stats_skopt, heatmap, optimize_result = backtest.optimize(
    n1 = [10, 100],      # Note: For method="skopt", we
    n2 = [20, 200],      # only need interval end-points
    n_enter = [10, 40],
    n_exit = [10, 30],
    constraint = lambda p: p.n_exit < p.n_enter < p.n1 < p.n2,
    maximize = 'Equity Final [$]',
    method = 'skopt',
    max_tries = 200,
    random_state = 0,
    return_heatmap = True,
    return_optimization = True)

In [ ]:
heatmap.sort_values().iloc[-3:]

In [ ]:
from skopt.plots import plot_objective

_ = plot_objective(optimize_result, n_points=10)

In [ ]:
from skopt.plots import plot_evaluations

_ = plot_evaluations(optimize_result, bins=10)

### Example 6: Trading with Machine Learning Models
- https://kernc.github.io/backtesting.py/doc/examples/Trading%20with%20Machine%20Learning.html